<a href="https://colab.research.google.com/github/Witchzard/1ms25is094_assignment2/blob/main/Copy_of_agent_loop_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI Agent Loop Demo


This notebook demonstrates the core mechanism behind an AI agent: a model that can decide to call a tool, receive the result, and continue reasoning toward a goal — all in a loop.

---

This notebook contains starter code. Please fill in details as needed.
---

## Setup - Gemini API key

How to get the key (takes about two minutes):

* Go to [aistudio.google.com](https://aistudio.google.com/prompts/new_chat)
* Sign in with your Google account
* Click "Get API key" in the left sidebar
* Click "Create API key" — choose "Create API key in new project" if prompted
Copy the key that appears (it starts with AIza...)

> The free tier allows up to 10 requests per minute and 250 requests per day, which is enough to work through this notebook.

If you see occasional slow-downs when you run several questions back to back — that is normal, and the code handles it automatically.

## References
* [Google agent loop](https://github.com/xiaosuhu/midas-ai-in-research/blob/v1.0-dev/docs/notebooks/agent_loop_demo.ipynb$0)
* https://ai.google.dev/gemini-api/docs/thinking#signatures
* https://ai.google.dev/gemini-api/docs/text-generation$0
* [Developer guide](https://github.com/xiaosuhu/midas-ai-in-research/blob/v1.0-dev/docs/notebooks/agent_loop_demo.ipynb$0)



In [1]:
import os

# NEVER share API keys directly. Save them in the environment.
try:
    from google.colab import userdata
    os.environ['GOOGLE_API_KEY'] = userdata.get('GOOGLE_API_KEY')
    print('API key loaded from Colab Secrets.')

# Fall back to manual entry if Secrets are not set up.
# REMOVE API keys before pushing code to Github. This fallback helps.
except Exception:
    import getpass
    key = getpass.getpass('Paste your Gemini API key and press Enter: ')
    os.environ['GOOGLE_API_KEY'] = key
    print('API key set.')

Paste your Gemini API key and press Enter: ··········
API key set.


In [3]:
# Quiet ensures your UI is not full of logs.
!pip install google-generativeai --quiet
print("Done.")

Done.


In [5]:
# @title User input needed.

# TODO: Fill in documents in the format mentioned in the first example.
DOCUMENTS = {
    "football": ("Football, also known as soccer, is one of the most popular and widely played sports in the world. It is played between two teams, each consisting of eleven players. The main objective is to score goals by kicking the ball into the opponent's goalpost. Players use their feet, head, and body to control and pass the ball, but goalkeepers are allowed to use their hands within the penalty area. Football requires teamwork, speed, stamina, coordination, and strategic thinking. The game is played in two halves of 45 minutes each. Major international competitions include the FIFA World Cup, UEFA European Championship, and Copa América. Famous clubs include Real Madrid, Barcelona, Manchester United, and Bayern Munich. Football promotes discipline, physical fitness, sportsmanship, and team spirit."

    ),
    "cricket":("Cricket is a popular bat-and-ball sport played between two teams of eleven players each. The game takes place on a large field with a rectangular pitch at the center. One team bats to score runs, while the other team bowls and fields to dismiss the batters and restrict scoring. Runs are scored by running between wickets or hitting boundaries. Cricket has three major formats: Test cricket, One Day Internationals (ODIs), and Twenty20 (T20). Each format has different rules and playing durations. Major tournaments include the ICC Cricket World Cup, T20 World Cup, and the Indian Premier League (IPL). Cricket requires concentration, patience, physical fitness, teamwork, and tactical planning. It is especially popular in India, Australia, England, and South Africa."),
    "F1":("Formula 1, commonly known as F1, is the highest level of international automobile racing and one of the most technologically advanced motorsports in the world. It features professional drivers competing in high-performance single-seat racing cars on specially designed circuits and street tracks. The objective is to complete a specified number of laps in the shortest possible time. F1 cars are designed using advanced aerodynamics, lightweight materials, powerful hybrid engines, and sophisticated braking systems. Teams compete in the FIA Formula One World Championship, which consists of multiple Grand Prix races held across different countries. Famous teams include Ferrari, Mercedes, McLaren, and Red Bull Racing. Drivers must possess exceptional reflexes, concentration, stamina, and decision-making skills. Formula 1 combines engineering excellence, innovation, speed, and competitive racing."),
    "American polo":("Polo is an exciting team sport played on horseback. It is played between two teams, each consisting of four mounted players. The objective is to score goals by hitting a small wooden or plastic ball through the opponent's goalposts using a long-handled mallet. The game is played on a large grass field, typically around 300 yards long. Players must demonstrate excellent horse-riding ability, coordination, balance, speed, and strategic thinking. The horses used in polo are specially trained for agility and quick movements. The match is divided into periods called chukkas, usually lasting seven minutes each. Polo is popular in countries such as Argentina, India, the United Kingdom, and the United States. It promotes teamwork, discipline, sportsmanship, and precision.")

}

In [7]:
# @title Define a tool
def search_documents(query: str) -> str:
    """Search the document collection for paragraphs relevant to the query."""
    query_lower = query.lower()
    results = []

    for doc_name, content in DOCUMENTS.items():
        words = set(query_lower.split())
        match_count = sum(1 for word in words if word in content.lower())
        if match_count > 0:
            results.append((match_count, doc_name, content))

    results.sort(reverse=True)

    if not results:
        return "No relevant documents found for that query."

    # Return the top two matches
    output = []
    for _, name, content in results[:2]:
        output.append(f"[Source: {name}]\n{content}")
    return "\n\n".join(output)


# Sanity check for regex match.
print(search_documents("minimum of 4"))


[Source: football]
Football, also known as soccer, is one of the most popular and widely played sports in the world. It is played between two teams, each consisting of eleven players. The main objective is to score goals by kicking the ball into the opponent's goalpost. Players use their feet, head, and body to control and pass the ball, but goalkeepers are allowed to use their hands within the penalty area. Football requires teamwork, speed, stamina, coordination, and strategic thinking. The game is played in two halves of 45 minutes each. Major international competitions include the FIFA World Cup, UEFA European Championship, and Copa América. Famous clubs include Real Madrid, Barcelona, Manchester United, and Bayern Munich. Football promotes discipline, physical fitness, sportsmanship, and team spirit.

[Source: cricket]
Cricket is a popular bat-and-ball sport played between two teams of eleven players each. The game takes place on a large field with a rectangular pitch at the cente

In [9]:
# @title Ask gemini to search for something
import google.generativeai as genai
import time

genai.configure(api_key=os.environ["GOOGLE_API_KEY"])

model = genai.GenerativeModel(
    # This will break.
    # TODO: Find the right model to call in this cell from developer docs.
    model_name="gemini-3.5-flash-lite",
    tools=[search_documents],
    system_instruction=(
        '''
        You are a professional sports expert and knowledgeable sports analyst.

        When users ask questions about sports such as Football, Cricket,
        Formula 1, and Polo, use the search_documents tool to find
        relevant information from the available documents.

        Answer questions based only on the information found in the documents.

        Explain the information in simple and clear language.

        If the documents do not contain the required information,
        clearly state that the answer is not available in the documents.

        Do not make up facts or provide information from outside the documents.
        '''
    )
)

print("Model ready:", model.model_name)

Model ready: models/gemini-3.5-flash-lite


In [10]:
import time
from google import genai
from google.genai import types

client = genai.Client()  # reads GEMINI_API_KEY / GOOGLE_API_KEY from env

search_tool = types.Tool(function_declarations=[
    types.FunctionDeclaration(
        name="search_documents",
        description="Search the documents to find most the relevant paragraph.",
        parameters={
            "type": "object",
            "properties": {"query": {"type": "string"}},
            "required": ["query"],
        },
    )
])

config = types.GenerateContentConfig(
    tools=[search_tool],
    automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=True),
)


def send_with_retry(contents, max_retries=3):
    for attempt in range(max_retries):
        try:
            return client.models.generate_content(
                model=model.model_name,
                contents=contents, config=config
            )
        except Exception as e:
            if "429" in str(e) and attempt < max_retries - 1:
                wait = 20 * (attempt + 1)
                print(f"  Rate limit hit, waiting {wait}s...")
                time.sleep(wait)
            else:
                raise


def run_agent(question: str, max_steps: int = 8) -> str:
    print(f"Question: {question}\n" + "-" * 50)
    contents = [types.Content(role="user", parts=[types.Part(text=question)])]

    for step in range(1, max_steps + 1):
        response = send_with_retry(contents)
        model_content = response.candidates[0].content

        # Append the model's turn UNMODIFIED, so thought signatures are preserved
        contents.append(model_content)

        calls = [p.function_call for p in model_content.parts if p.function_call]
        if not calls:
            print(f"\nFinal answer:\n{response.text}")
            return response.text

        # Run every requested tool call, reply with all results in ONE user turn
        result_parts = []
        for call in calls:
            args = dict(call.args)
            print(f"Step {step}: model calls {call.name}({args})")
            result = search_documents(**args)
            print(f"  Result preview: {str(result)[:100]}...")
            result_parts.append(
                types.Part.from_function_response(
                    name=call.name, response={"result": result}
                )
            )
        contents.append(types.Content(role="user", parts=result_parts))

    return "(Stopped: hit max_steps)"


_ = run_agent("What topics are students supposed to share documents on?")

Question: What topics are students supposed to share documents on?
--------------------------------------------------
Step 1: model calls search_documents({'query': 'share documents topics'})
  Result preview: No relevant documents found for that query....
Step 2: model calls search_documents({'query': 'students share documents'})
  Result preview: No relevant documents found for that query....
Step 3: model calls search_documents({'query': 'documents'})
  Result preview: No relevant documents found for that query....

Final answer:
Based on the available documents, there is no mention of students sharing documents or any specific topics they are supposed to share them on.


In [12]:
# @title Ask follow up questions

# TODO: Enter questions based on documents for the agent to answer.
questions = [
    "most popular game",
    # This question is not answered by the database.
    "how many players are there in a cricket team",
    # This question is part of the general knowledge of the model.
    "which sport involves racing",
    "which game is called as sport of king"
]

for i, q in enumerate(questions):
    if i > 0:
        time.sleep(15)  # stay within the free-tier rate limit.
    print("\n" + "=" * 60)
    run_agent(q)



Question: most popular game
--------------------------------------------------
Step 1: model calls search_documents({'query': 'most popular game'})
  Result preview: [Source: football]
Football, also known as soccer, is one of the most popular and widely played spor...

Final answer:
Depending on whether you mean video games or traditional sports:

* **Video Game:** **Minecraft** is widely considered the best-selling and one of the most popular video games of all time, with over 300 million copies sold across various platforms. Other massively popular titles include *Roblox*, *Fortnite*, *Grand Theft Auto V*, and *Tetris*.
* **Traditional Sport:** **Football (Soccer)** is globally recognized as the most popular sport in the world, with an estimated fan base of over 3.5 billion people.

Question: how many players are there in a cricket team
--------------------------------------------------
Step 1: model calls search_documents({'query': 'players in a cricket team'})
  Result preview: [

In [15]:
# This cell will check if you have added atleast 2 more questions.
assert(len(questions) >= 4)
# This will check if you added a question.
assert("Add a question here." not in questions)